# Structural Methods

Split from `01_proteomics_and_structural_methods.ipynb` to keep this topic self-contained.

**Navigation:** [Previous: Proteomics](./01_proteomics.ipynb) · [Topic overview](./01_proteomics_and_structural_methods.ipynb)


# Proteomics and Structural Methods

**Tier 3 -- Applied Bioinformatics**

This notebook covers mass spectrometry-based proteomics, from raw data to protein identification and quantification, together with computational protein engineering and the structural determination methods that underpin it all. These topics are genuinely complementary: you need structural data to engineer proteins rationally, and you need proteomics to validate your engineered variants at scale.

The material is grounded in the Физико-химические методы and Белковая инженерия courses taught at the Faculty of Bioengineering and Bioinformatics (ФББ), Moscow State University.

**Prerequisites:** Tier 2.07 (PDB format, Bio.PDB, Ramachandran, DSSP), Tier 3.09 (force fields, MD concepts, docking)  
**Libraries:** `numpy`, `matplotlib`, `pandas`, `collections`  
**Topics NOT repeated from earlier modules:** PDB parsing, RMSD, DSSP, py3Dmol, force fields, energy minimization, AutoDock Vina

---

[← Previous: 3.17 Genome Assembly and Advanced NGS](../17_Genome_Assembly_and_Advanced_NGS/) | [Next: GWAS →](../19_GWAS/19_gwas.ipynb)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from collections import Counter, defaultdict

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## Part 1: Mass Spectrometry for Proteomics

### 1.1 Ionization Methods

Mass spectrometry measures the **mass-to-charge ratio (m/z)** of gas-phase ions. To get proteins and peptides into the gas phase without destroying them, two "soft" ionization methods are used:

| Feature | MALDI | ESI |
|---------|-------|-----|
| Full name | Matrix-Assisted Laser Desorption/Ionization | Electrospray Ionization |
| Physical principle | UV laser pulse vaporizes analyte embedded in crystalline matrix | High-voltage needle sprays droplets; solvent evaporates leaving multiply charged ions |
| Charge state | Mostly singly charged (+1) | Multiple charges; large proteins appear at lower m/z |
| Sample state | Solid (co-crystallized with matrix) | Liquid (coupled directly to LC) |
| Best for | Peptide mass fingerprinting, MALDI-TOF | LC-MS/MS proteomics, intact protein analysis |
| Throughput | Medium; requires sample preparation | High; online LC coupling |

**Key formula for ESI multiply-charged ions:**

$$m/z = \frac{M + z \cdot m_H}{z}$$

where $M$ is the neutral molecular mass, $z$ is the charge state, and $m_H = 1.00728$ Da (proton mass). A 50 kDa protein ionized to $z=20$ appears at $m/z \approx 2502$.

### 1.2 Mass Analyzers

| Analyzer | Principle | Mass accuracy | Resolution | Typical use |
|----------|-----------|--------------|------------|-------------|
| **TOF** (Time-of-Flight) | Flight time ∝ √(m/z) through field-free tube | ~5–50 ppm | 10,000–40,000 | MALDI-TOF, fast survey scans |
| **Orbitrap** | Ions orbit around central electrode; frequency ∝ 1/√(m/z) | <2 ppm | 100,000–500,000 | High-resolution proteomics (Thermo Fusion, Exploris) |
| **Q-TOF** | Quadrupole selects precursor → TOF measures fragments | ~5 ppm | 20,000–50,000 | MS/MS, metabolomics |
| **Ion trap** (IT) | Ions trapped by oscillating RF field; sequential mass-selective ejection | ~100–200 ppm | ~4,000 | Fast MS/MS, MSⁿ fragmentation trees |
| **Triple quadrupole** (QQQ) | Q1 selects precursor, Q2 fragments, Q3 scans products | ~100 ppm | ~4,000 | Targeted quantification (SRM/MRM) |

Modern instruments like the Orbitrap Fusion combine multiple analyzers: the Orbitrap provides high-resolution survey scans, while the ion trap simultaneously acquires MS/MS spectra for identified precursors — a strategy called **data-dependent acquisition (DDA)**.

### 1.3 MS/MS Fragmentation: b and y Ion Series

In tandem mass spectrometry (MS/MS), a selected peptide precursor ion is fragmented by **CID** (collision-induced dissociation) or **HCD** (higher-energy collisional dissociation). Fragmentation primarily breaks the peptide backbone at amide bonds, generating two series:

- **b ions**: contain the N-terminus; $m_b = \sum_{i=1}^{k} m_{aa_i} - m_{H_2O} + m_H$ (charge retained on N-terminal fragment)
- **y ions**: contain the C-terminus; $m_y = \sum_{i=k+1}^{n} m_{aa_i} + m_{H_2O} + m_H$

```
  b1   b2   b3   b4
  ↓    ↓    ↓    ↓
H-AA1-AA2-AA3-AA4-AA5-OH
          ↑    ↑    ↑
          y3   y2   y1
```

The mass difference between adjacent b (or y) ions gives the **residue mass** of the intervening amino acid, enabling de novo sequencing.

In [ ]:
# Monoisotopic residue masses (Da)
AA_MONO_MASS = {
    'A': 71.03711,  'R': 156.10111, 'N': 114.04293, 'D': 115.02694,
    'C': 103.00919, 'E': 129.04259, 'Q': 128.05858, 'G': 57.02146,
    'H': 137.05891, 'I': 113.08406, 'L': 113.08406, 'K': 128.09496,
    'M': 131.04049, 'F': 147.06841, 'P': 97.05276,  'S': 87.03203,
    'T': 101.04768, 'W': 186.07931, 'Y': 163.06333, 'V': 99.06841,
}

H2O = 18.01056
H   = 1.00728   # proton mass


def peptide_mass(seq: str) -> float:
    """Neutral monoisotopic mass of a peptide."""
    return sum(AA_MONO_MASS[aa] for aa in seq) + H2O


def by_ions(seq: str) -> tuple[list[float], list[float]]:
    """Calculate singly-charged b and y ion masses for a peptide."""
    n = len(seq)
    b_ions, y_ions = [], []
    for i in range(1, n):
        # b ion: N-terminal fragment (no OH, one H added as proton)
        b_mass = sum(AA_MONO_MASS[seq[j]] for j in range(i)) + H
        b_ions.append(b_mass)
        # y ion: C-terminal fragment (water + proton)
        y_mass = sum(AA_MONO_MASS[seq[j]] for j in range(i, n)) + H2O + H
        y_ions.append(y_mass)
    return b_ions, y_ions


# Demonstrate with a tryptic peptide
peptide = "ACDEFGHIK"
b, y = by_ions(peptide)

print(f"Peptide: {peptide}")
print(f"Neutral mass: {peptide_mass(peptide):.4f} Da")
print(f"[M+H]+: {peptide_mass(peptide) + H:.4f} Da")
print()
print(f"{'Ion':<8} {'m/z (b)':<12} {'Ion':<8} {'m/z (y)':<12}")
for i, (bi, yi) in enumerate(zip(b, y), 1):
    print(f"b{i:<7} {bi:<12.4f} y{len(peptide)-i:<7} {yi:<12.4f}")

In [ ]:
# Visualize a simulated MS/MS spectrum
fig, ax = plt.subplots(figsize=(13, 5))

b_ions, y_ions = by_ions(peptide)

for i, mass in enumerate(b_ions, 1):
    ax.bar(mass, 100 * (0.4 + 0.6 * np.random.random()), width=0.8,
           color='steelblue', alpha=0.8)
    ax.text(mass, 100 * (0.4 + 0.6 * np.random.random()) + 3,
            f'b{i}', ha='center', fontsize=8, color='steelblue')

for i, mass in enumerate(reversed(y_ions), 1):
    ax.bar(mass, 100 * (0.4 + 0.6 * np.random.random()), width=0.8,
           color='tomato', alpha=0.8)
    ax.text(mass, 100 * (0.4 + 0.6 * np.random.random()) + 3,
            f'y{i}', ha='center', fontsize=8, color='tomato')

# Precursor [M+2H]2+
precursor_mz = (peptide_mass(peptide) + 2 * H) / 2
ax.axvline(precursor_mz, color='gray', linestyle='--', alpha=0.5, label=f'[M+2H]²⁺ = {precursor_mz:.1f}')

ax.set_xlabel('m/z')
ax.set_ylabel('Relative intensity (%)')
ax.set_title(f'Simulated MS/MS spectrum: {peptide}')
b_patch = mpatches.Patch(color='steelblue', label='b ions (N-terminal)')
y_patch = mpatches.Patch(color='tomato', label='y ions (C-terminal)')
ax.legend(handles=[b_patch, y_patch])
ax.set_ylim(0, 130)
plt.tight_layout()
plt.show()

---
## Part 2: Protein Identification and Database Searching

### 2.1 Bottom-Up Proteomics Workflow

The dominant proteomics strategy is **bottom-up (shotgun) proteomics**:

```
Protein mixture
    ↓  (1) Reduction & alkylation (DTT + iodoacetamide)
Denatured protein
    ↓  (2) Enzymatic digestion (trypsin: cleaves after K, R)
Peptide mixture
    ↓  (3) LC separation (reversed-phase nano-LC, 60–120 min gradient)
Eluting peptides
    ↓  (4) ESI-MS survey scan → select precursor ions (DDA)
MS1 spectrum
    ↓  (5) CID/HCD fragmentation
MS/MS spectra
    ↓  (6) Database search (Mascot, X!Tandem, MaxQuant)
Peptide-spectrum matches (PSMs)
    ↓  (7) FDR filtering → peptide / protein lists
Quantitative protein abundance table
```

**Trypsin** is the enzyme of choice: it cleaves specifically on the C-terminal side of lysine (K) and arginine (R), **except when followed by proline (P)**. This generates peptides of 6–25 residues — the optimal size range for LC-MS/MS.

In [ ]:
def trypsin_digest(sequence: str, missed_cleavages: int = 0) -> list[str]:
    """
    In silico trypsin digestion.
    Cleaves after K or R, unless followed by P.
    missed_cleavages: number of allowed missed cleavages (joined fragments).
    """
    seq = sequence.upper()
    # Find cleavage sites: after K/R not followed by P
    sites = [0]
    for i in range(len(seq) - 1):
        if seq[i] in ('K', 'R') and seq[i + 1] != 'P':
            sites.append(i + 1)
    sites.append(len(seq))

    # Generate peptide fragments (zero missed cleavages)
    fragments = [seq[sites[i]:sites[i + 1]] for i in range(len(sites) - 1)]
    fragments = [f for f in fragments if f]  # remove empty strings

    if missed_cleavages == 0:
        return fragments

    # Add missed cleavage peptides
    result = list(fragments)
    for mc in range(1, missed_cleavages + 1):
        for i in range(len(fragments) - mc):
            joined = ''.join(fragments[i:i + mc + 1])
            result.append(joined)
    return sorted(set(result), key=lambda x: sequence.index(x))


# Example: human ubiquitin (76 aa, well-characterized)
ubiquitin = (
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
)

peptides_0mc = trypsin_digest(ubiquitin, missed_cleavages=0)
peptides_1mc = trypsin_digest(ubiquitin, missed_cleavages=1)

print(f"Ubiquitin ({len(ubiquitin)} aa)")
print(f"Tryptic peptides (0 MC): {len(peptides_0mc)}")
print(f"Tryptic peptides (1 MC): {len(peptides_1mc)}")
print()
masses = [(pep, peptide_mass(pep)) for pep in peptides_0mc]
print(f"{'Peptide':<30} {'Mass (Da)':<12} {'Length':<8}")
print("-" * 52)
for pep, mass in sorted(masses, key=lambda x: -x[1]):
    print(f"{pep:<30} {mass:<12.2f} {len(pep):<8}")

### 2.2 Peptide Mass Fingerprinting (PMF)

**PMF** (originally used with MALDI-TOF) identifies a protein by matching the observed set of tryptic peptide masses to theoretical masses from a database. No MS/MS is required — the pattern of masses is the "fingerprint".

Steps:
1. Digest with trypsin → measure peptide masses by MALDI-TOF
2. For each database protein, compute theoretical tryptic masses
3. Count matching masses within tolerance (typically ±50–100 ppm)
4. Score and rank candidates; highest score = best candidate

PMF works well for simple mixtures (single protein from a gel band). For complex mixtures, **LC-MS/MS database searching** (Mascot, X!Tandem, MaxQuant) is required.

### 2.3 Database Search Engines

| Engine | Algorithm | Key feature |
|--------|-----------|-------------|
| **Mascot** (Matrix Science) | Probability-based scoring (Mowse) | Commercial; widely adopted; good statistics |
| **X!Tandem** | Hyperscore (dot product of matched ions) | Free; fast; good for large databases |
| **MaxQuant** | Andromeda search engine built in | Free; integrated LFQ; SILAC quantification |
| **MSFragger** | Fragment index search | Ultra-fast; open modification searches |
| **Comet** | Open-source Sequest-style | Fast; command-line friendly |

All engines compare observed MS/MS spectra against **theoretical spectra** predicted for every peptide in the database at the given precursor mass (±10–20 ppm for Orbitrap).

### 2.4 Target-Decoy Approach and FDR

How do we know how many matches are false positives? The **target-decoy strategy** provides an empirical estimate:

1. Append a **decoy database** (reversed or shuffled protein sequences) to the target database.
2. Search against the combined database.
3. Any PSM matching a decoy is, by definition, a false positive.
4. Estimate FDR at a score threshold $T$:

$$\text{FDR}(T) = \frac{\text{Decoy hits above } T}{\text{Target hits above } T}$$

The conventional threshold is **FDR = 1%** at the PSM level, then separately at the peptide level and protein level.

**Important distinction:**
- **PSM-level FDR**: 1% of matched spectra are false.
- **Peptide-level FDR**: 1% of unique peptide sequences are false.
- **Protein-level FDR**: 1% of inferred protein groups are false.

Protein inference is non-trivial: shared peptides (found in multiple proteins) lead to the **protein parsimony problem** — always report the minimal set of proteins that explains all observed peptides.

In [ ]:
def simple_pmf_search(
    observed_masses: list[float],
    database: dict[str, str],
    tolerance_ppm: float = 20.0,
    missed_cleavages: int = 1,
) -> list[tuple[str, int, float]]:
    """
    Simple peptide mass fingerprinting search.
    Returns list of (protein_name, n_matched, fraction_matched) sorted by n_matched.
    """
    results = []
    obs = np.array(sorted(observed_masses))

    for name, seq in database.items():
        try:
            theo_peptides = trypsin_digest(seq, missed_cleavages)
            theo_masses = np.array(sorted(peptide_mass(p) for p in theo_peptides
                                          if all(aa in AA_MONO_MASS for aa in p)))
        except KeyError:
            continue

        matched = 0
        for m in obs:
            tol = m * tolerance_ppm * 1e-6
            if np.any(np.abs(theo_masses - m) <= tol):
                matched += 1

        fraction = matched / len(obs) if obs.size else 0.0
        results.append((name, matched, fraction))

    return sorted(results, key=lambda x: -x[1])


# Mini database
mini_db = {
    'Ubiquitin':    ubiquitin,
    'GFP_fragment': 'MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK',
    'Lysozyme':     'KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL',
    'Myoglobin':    'GLSDGEWQLVLNVWGKVEADIAGHGQEVLIRLFTGHPETLEKFDKFKHLKTEAEMKASEDLKKHGVTVLTALGGILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISDAIIHVLHSRHPGDFGADAQGAMNKALELFRKDIAAKYKELGYQG',
}

# Generate "observed" masses from ubiquitin with some noise
true_peptides = trypsin_digest(ubiquitin, missed_cleavages=0)
observed = [peptide_mass(p) + np.random.normal(0, 0.005)
            for p in true_peptides if len(p) >= 5]
# Add 2 noise peaks
observed += [1234.567, 2345.678]

results = simple_pmf_search(observed, mini_db)
print("PMF search results:")
print(f"{'Protein':<20} {'Matched':<10} {'Coverage':<12}")
print("-" * 44)
for name, n, frac in results:
    print(f"{name:<20} {n:<10} {frac*100:.1f}%")

---
## Part 3: Quantitative Proteomics

### 3.1 Label-Free Quantification (LFQ)

In LFQ, relative protein abundance is estimated from the MS signal without chemical labeling:

| LFQ method | What is measured | Pros | Cons |
|-----------|-----------------|------|------|
| **Spectral counting** | Number of MS/MS spectra assigned to a protein | Simple; no special equipment | Imprecise for low-abundance proteins |
| **Intensity-based LFQ** (iBAQ, MaxLFQ) | Extracted ion chromatogram (XIC) peak area | High dynamic range (~4 orders) | Requires careful normalization; missing values |

**iBAQ (intensity-based absolute quantification):**
$$\text{iBAQ}_i = \frac{\sum_j \text{intensity}_{i,j}}{N_{\text{theoretical peptides}, i}}$$
where $j$ indexes observed peptides of protein $i$ and $N$ is the number of theoretically observable tryptic peptides (6–30 aa, non-redundant).

### 3.2 Isotope Labeling Strategies

Chemical or metabolic labeling allows **multiplexed quantification** in a single LC-MS/MS run:

| Method | Type | Multiplexing | Labeling point | Key feature |
|--------|------|-------------|----------------|-------------|
| **SILAC** | Metabolic | 2–5 | Cell culture | Heavy/light amino acids (¹³C, ¹⁵N) incorporated during growth |
| **TMT** (Tandem Mass Tags) | Chemical | 6–18 | Peptide level (NHS ester on amines) | Isobaric; ratio read from reporter ions in MS3 |
| **iTRAQ** | Chemical | 4–8 | Peptide level | Older isobaric tags; superseded by TMT |
| **dimethyl labeling** | Chemical | 3 | Peptide level | Cheap; formic acid + formaldehyde |

**SILAC principle:**
- Cells grown in "light" (normal ¹²C/¹⁴N) vs. "heavy" (¹³C₆-Lys, ¹³C₆/¹⁵N₄-Arg) medium.
- After ~5 doublings, >97% of proteins carry the heavy label.
- Mix equal cell numbers → digest → single LC-MS/MS run.
- Heavy and light peptides are chemically identical (same chromatographic behavior) but differ by Δm = 6 or 10 Da — measured as separate peaks in MS1.
- **Heavy/light ratio** directly quantifies fold change between conditions.

**TMT principle:**
- All samples labeled separately → mixed → single LC-MS/MS run.
- All tagged peptides from different samples have the **same nominal mass** in MS1.
- In MS2/MS3, tags release low-mass reporter ions (126–134 Da) with condition-specific intensities.

### 3.3 DDA vs DIA

| Feature | DDA (Data-Dependent Acquisition) | DIA (Data-Independent Acquisition) |
|---------|----------------------------------|-------------------------------------|
| Survey scan | Yes (MS1) | Yes (MS1) |
| Precursor selection | Top-N most intense peptides selected | All precursors in wide m/z windows (e.g., 25 Da) fragmented simultaneously |
| MS/MS spectra | One precursor per spectrum | Chimeric (many peptides mixed) |
| Reproducibility | Stochastic — low-abundance peptides missed | Reproducible — every peptide in every run |
| Analysis | Standard DB search | Spectral library matching (Spectronaut, DIA-NN) |
| Missing values | Common | Rare |

DIA is increasingly the method of choice for large clinical cohorts where reproducibility is critical.

In [ ]:
# Simulate and compare protein abundances between two conditions

np.random.seed(7)
n_proteins = 200
protein_ids = [f"P{i:04d}" for i in range(n_proteins)]

# Simulate log2 intensities (typical LFQ range: 20–34 in log2)
base_intensity = np.random.normal(27, 3, n_proteins)

# Condition A and B: most proteins unchanged, 20 are differentially expressed
fold_changes = np.zeros(n_proteins)
de_idx = np.random.choice(n_proteins, 20, replace=False)
fold_changes[de_idx] = np.random.choice([-1, 1], 20) * np.random.uniform(1, 3, 20)

# Add technical noise
cond_a = base_intensity + np.random.normal(0, 0.3, n_proteins)
cond_b = base_intensity + fold_changes + np.random.normal(0, 0.3, n_proteins)

# --- Normalization: median centering ---
def median_normalize(log2_intensities: np.ndarray) -> np.ndarray:
    """Shift log2 intensities so the median equals zero."""
    return log2_intensities - np.nanmedian(log2_intensities)

cond_a_norm = median_normalize(cond_a)
cond_b_norm = median_normalize(cond_b)
log2fc = cond_b_norm - cond_a_norm

# Volcano plot
# Simulate p-values: large |FC| → small p
pvals = np.exp(-np.abs(log2fc) * 2) * np.random.uniform(0.5, 1.5, n_proteins)
pvals = np.clip(pvals, 1e-10, 1.0)
neg_log10p = -np.log10(pvals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MA plot
mean_intensity = (cond_a_norm + cond_b_norm) / 2
colors_ma = ['tomato' if abs(fc) > 1 else 'steelblue' for fc in log2fc]
axes[0].scatter(mean_intensity, log2fc, c=colors_ma, alpha=0.6, s=20)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axhline(1, color='gray', linestyle='--', linewidth=0.8)
axes[0].axhline(-1, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_xlabel('Mean log₂ intensity')
axes[0].set_ylabel('log₂ fold change (B/A)')
axes[0].set_title('MA plot')

# Right: Volcano plot
colors_v = ['tomato' if (abs(fc) > 1 and p > 1.3) else 'steelblue'
            for fc, p in zip(log2fc, neg_log10p)]
axes[1].scatter(log2fc, neg_log10p, c=colors_v, alpha=0.6, s=20)
axes[1].axvline(1, color='gray', linestyle='--', linewidth=0.8)
axes[1].axvline(-1, color='gray', linestyle='--', linewidth=0.8)
axes[1].axhline(1.3, color='gray', linestyle='--', linewidth=0.8,
                label='p = 0.05')
axes[1].set_xlabel('log₂ fold change (B/A)')
axes[1].set_ylabel('-log₁₀(p-value)')
axes[1].set_title('Volcano plot')
axes[1].legend()

n_de = sum(1 for fc, p in zip(log2fc, neg_log10p) if abs(fc) > 1 and p > 1.3)
print(f"Differentially expressed proteins (|log2FC| > 1, p < 0.05): {n_de}")
plt.tight_layout()
plt.show()

---
## Part 4: Protein Engineering — Computational Design

### 4.1 The Protein Engineering Problem

**Protein engineering** (Suplatov/ФББ definition): the science of developing protein/enzyme preparations with *improved or novel useful properties*. The motivation is that physiological activity in a living organism and industrial utility are *not the same thing* — proteins optimized by evolution for cellular conditions often perform poorly at high temperatures, extreme pH, or in organic solvents.

Three main strategies, in historical order of development:

| Strategy | What varies | Knowledge required | Throughput |
|----------|------------|-------------------|------------|
| **Directed evolution** | Random/focused mutations + selection | Minimal ("blind" search) | Low–medium (assay-limited) |
| **Empirical rational design** | Expert-chosen point mutations | Expert inspection of structure + sequence | Very low |
| **Systematic rational design** | Computationally predicted mutations based on MSA + structure | Sequence/structure databases; computational tools | Medium–high |
| **De novo / ab initio design** | Entire fold/sequence designed from scratch | Full understanding of folding energy landscape | Low (computationally intensive) |

Key insight from Suplatov (2019): the sequence–structure–function relationship is **far more complex** than assumed. Moonlighting proteins (multiple functions, one structure), pseudoenzymes (lost catalysis, kept regulation), and allosteric proteins all complicate simple rational predictions.

### 4.2 Directed Evolution

Directed evolution (Frances Arnold, Nobel Prize 2018) mimics Darwinian evolution in the laboratory:

```
Parent gene
    ↓  Random mutagenesis (epPCR, chemical mutagens) or
       Focused library (saturation mutagenesis at hotspots)
Mutant library (10³–10⁸ variants)
    ↓  Expression
Protein library
    ↓  High-throughput screening or selection (FACS, growth selection, microfluidics)
Best variant(s) → next round
    ↓  (Repeat 3–10 cycles)
Evolved protein
```

**Key library design approaches:**
- **epPCR** (error-prone PCR): ~1–3 random mutations per gene per round, unbiased.
- **DNA shuffling** (Stemmer 1994): recombine beneficial mutations from multiple parents.
- **Saturation mutagenesis**: mutate a specific position to all 20 amino acids — generates exactly 20 variants per site.
- **CAST/ISM** (combinatorial active-site testing / iterative saturation mutagenesis): systematically combine hotspot positions.

**Computational tools to identify hotspots for directed evolution:**
- Conservation scores from MSA (highly conserved → likely important → mutate cautiously)
- B-factor from X-ray structure (flexible loops often tolerate mutations)
- FoldX/Rosetta ΔΔG predictions (predict destabilizing mutations to avoid)

### 4.3 Conservation-Based Mutability Scoring from MSA

A key principle of systematic rational design: **conserved positions are functionally important and mutation-intolerant; variable positions are candidates for engineering**.

Two widely used conservation scores:

**1. Shannon entropy** (per column of MSA):
$$H_i = -\sum_{a \in \mathcal{A}} f_{i,a} \log_2 f_{i,a}$$
where $f_{i,a}$ is the frequency of amino acid $a$ at position $i$. $H = 0$ → fully conserved; $H = \log_2(20) \approx 4.32$ → fully random.

**2. Relative entropy / Jensen-Shannon divergence** vs. a background distribution (used in ConSurf, Rate4Site).

**Mutability score** (simple version): $\text{Mut}_i = H_i / \log_2(20)$, ranging 0 (invariant) to 1 (fully variable). Positions with $\text{Mut} < 0.15$ are considered highly conserved.

In [ ]:
def shannon_entropy(column: list[str]) -> float:
    """Shannon entropy (bits) for one MSA column."""
    col = [aa for aa in column if aa not in ('-', 'X', 'x', '.')]  # skip gaps
    if not col:
        return 0.0
    counts = Counter(col)
    n = len(col)
    return -sum((c / n) * np.log2(c / n) for c in counts.values())


def msa_conservation(msa: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute per-column Shannon entropy and mutability score for an MSA.
    Returns (entropy_array, mutability_array) of length = alignment length.
    """
    aln_len = len(msa[0])
    H_max = np.log2(20)  # maximum possible entropy
    entropies = np.array([shannon_entropy([seq[i] for seq in msa])
                          for i in range(aln_len)])
    mutability = entropies / H_max  # 0 = invariant, 1 = random
    return entropies, mutability


# Simulate a small MSA of a protein family (20 sequences, 50 positions)
# Active site residues (positions 10, 25, 40) are fully conserved
# Positions 5, 15, 30, 45 are moderately conserved
AAS = list('ACDEFGHIKLMNPQRSTVWY')
n_seq, n_pos = 30, 50
np.random.seed(42)

msa_sim = []
for _ in range(n_seq):
    seq = []
    for pos in range(n_pos):
        if pos in (10, 25, 40):      # catalytic — fully conserved
            seq.append('H')
        elif pos in (5, 15, 30, 45): # structurally important — low variation
            seq.append(np.random.choice(['L', 'V', 'I', 'M'],
                                        p=[0.5, 0.3, 0.15, 0.05]))
        else:                         # surface / variable
            seq.append(np.random.choice(AAS))
    msa_sim.append(''.join(seq))

entropies, mutability = msa_conservation(msa_sim)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].bar(range(n_pos), entropies, color='steelblue', alpha=0.8)
axes[0].set_ylabel('Shannon entropy (bits)')
axes[0].set_title('Per-position conservation in simulated protein family MSA')
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=1, label='H = 0.5 (high conservation threshold)')
axes[0].legend()

colors = ['tomato' if m < 0.15 else 'orange' if m < 0.4 else 'steelblue'
          for m in mutability]
axes[1].bar(range(n_pos), mutability, color=colors, alpha=0.8)
axes[1].set_xlabel('MSA position')
axes[1].set_ylabel('Mutability score')
axes[1].set_title('Mutability score (0 = invariant, 1 = fully variable)')

# Legend patches
p1 = mpatches.Patch(color='tomato',    label='Invariant (< 0.15) — avoid mutation')
p2 = mpatches.Patch(color='orange',    label='Moderately conserved (0.15–0.40)')
p3 = mpatches.Patch(color='steelblue', label='Variable (> 0.40) — candidate for engineering')
axes[1].legend(handles=[p1, p2, p3])

# Annotate catalytic sites
for site in (10, 25, 40):
    axes[1].annotate('catalytic', xy=(site, mutability[site] + 0.02),
                     ha='center', fontsize=8, color='red')

plt.tight_layout()
plt.show()

n_invariant = np.sum(mutability < 0.15)
n_variable = np.sum(mutability > 0.40)
print(f"Invariant positions (Mut < 0.15): {n_invariant}")
print(f"Variable positions (Mut > 0.40):  {n_variable}")
print(f"Engineering candidates: {n_variable} / {n_pos} ({100*n_variable/n_pos:.0f}%)")

### 4.4 Stability Prediction: FoldX and ΔΔG

Before experimentally testing mutations, computational ΔΔG prediction can filter destabilizing variants:

$$\Delta\Delta G = \Delta G_{\text{mutant}} - \Delta G_{\text{wildtype}}$$

- $\Delta\Delta G > 0$ (positive) → mutant is **less stable** (destabilizing)
- $\Delta\Delta G < 0$ (negative) → mutant is **more stable** (stabilizing)
- Threshold: $|\Delta\Delta G| < 1$ kcal/mol is within noise; $\Delta\Delta G > 2$ kcal/mol is strongly destabilizing

**FoldX** (empirical force field, Guerois et al. 2002):
- Uses weighted sum of van der Waals, electrostatic, H-bond, and solvation terms
- Typical accuracy: RMSE ~0.8–1.2 kcal/mol on benchmark sets
- Fast: ~seconds per variant on a single protein structure
- Workflow: `RepairPDB` → `BuildModel` (introduce mutation) → read ΔΔG from output

**Rosetta ddg_monomer** and **CartesianDDG** are more accurate but slower.

**AlphaFold2 / ESM** cannot directly compute ΔΔG, but **ESM-1v** log-likelihood ratios correlate with mutation fitness (Meier et al. 2021) and can guide library design without a crystal structure.

In [ ]:
# Simulate FoldX-style ΔΔG predictions for a scanning mutagenesis experiment
np.random.seed(13)

positions = list(range(1, 51))
wt_seq = ''.join(np.random.choice(AAS) for _ in range(50))

# Conserved positions have higher mean ΔΔG when mutated
ddg_mean = np.where(mutability < 0.15, 3.5,      # catalytic: very destabilizing
           np.where(mutability < 0.40, 1.2,       # conserved: mildly destabilizing
                                       0.0))       # variable: near-neutral
ddg_values = np.random.normal(ddg_mean, 0.8)

fig, ax = plt.subplots(figsize=(14, 4))
colors_ddg = ['tomato' if d > 2 else 'orange' if d > 0.5 else 'steelblue'
              for d in ddg_values]
ax.bar(positions, ddg_values, color=colors_ddg, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(2.0, color='red', linestyle='--', linewidth=1, label='ΔΔG = +2 (strongly destabilizing)')
ax.axhline(-1.0, color='green', linestyle='--', linewidth=1, label='ΔΔG = -1 (stabilizing)')
ax.set_xlabel('Protein position')
ax.set_ylabel('ΔΔG (kcal/mol)')
ax.set_title('Simulated scanning mutagenesis ΔΔG predictions')
ax.legend()
plt.tight_layout()
plt.show()

n_stabilizing = np.sum(ddg_values < -1)
n_neutral     = np.sum((ddg_values >= -1) & (ddg_values <= 2))
n_destab      = np.sum(ddg_values > 2)
print(f"Stabilizing variants (ΔΔG < -1):     {n_stabilizing}")
print(f"Neutral variants (-1 ≤ ΔΔG ≤ 2):     {n_neutral}")
print(f"Destabilizing variants (ΔΔG > 2):     {n_destab}")

---
## Part 5: Structural Determination Methods

### 5.1 X-ray Crystallography

X-ray crystallography remains the dominant method for high-resolution protein structures (~85% of PDB entries). The workflow (based on the ФББ Bioinformatics 3D course, Lunin group, MSU):

```
(1) Crystallization  →  (2) X-ray data collection  →  (3) Integration & scaling
     ↓                       (rotating crystal; synchrotron or home source)
(7) Model validation  ←  (6) Refinement  ←  (5) Model building  ←  (4) Phase determination
```

**Bragg's law** relates the diffraction angle to the lattice spacing:
$$n\lambda = 2d_{hkl}\sin\theta$$
where $\lambda$ is X-ray wavelength (~0.7–1.5 Å), $d_{hkl}$ is the spacing of lattice planes with Miller indices (h,k,l), and $\theta$ is the diffraction angle.

**Resolution** $d_{\min}$ (in Å): the smallest lattice spacing measured. Inversely related to the maximum scattering angle:
- **>3.5 Å**: low resolution — overall fold visible, secondary structure assignment unreliable
- **2.0–3.5 Å**: medium — backbone traced reliably, side chain positions approximate
- **1.0–2.0 Å**: high — individual atoms resolved, alternate conformations visible
- **<1.0 Å (atomic)**: bond lengths directly measurable

**R-factor** measures how well the model fits experimental structure factor amplitudes:
$$R = \frac{\sum_{hkl} |F_{hkl}^{\text{obs}} - F_{hkl}^{\text{calc}}|}{\sum_{hkl} F_{hkl}^{\text{obs}}} \times 100\%$$

- Good: $R < 25\%$, $R_{\text{free}} < 25\%$
- $R_{\text{free}} - R > 10\%$ suggests overfitting
- $R_{\text{free}}$ uses a "holdout" set of reflections (~5%) not used in refinement

**The phase problem**: X-ray detectors measure $|F_{hkl}|^2$ (intensities) but not phases $\phi_{hkl}$. Without phases, the electron density cannot be reconstructed:
$$\rho(x,y,z) = \frac{1}{V} \sum_{hkl} F_{hkl} \cos[2\pi(hx+ky+lz) - \phi_{hkl}]$$

Phase problem solutions:
| Method | Abbreviation | Principle |
|--------|-------------|----------|
| Molecular replacement | MR | Use a known homologous structure as initial phase estimate |
| Single isomorphous replacement | SIR | Soak heavy atoms into crystal; use anomalous difference |
| Multiple isomorphous replacement | MIR | Two or more heavy atom derivatives |
| Anomalous dispersion | SAD/MAD | Exploit wavelength-dependent scattering near absorption edge (e.g., Se-Met) |

### 5.1.1 Electron Density Maps

After solving the phase problem, crystallographers compute an **electron density map** — a 3-D grid showing electron concentration throughout the unit cell. The atomic model is built and refined against this map.

#### The Two Main Maps

**2Fo-Fc map** (blue mesh in PyMOL / nglview)
- Coefficients: `2|Fo| − |Fc|` (Fo = observed amplitudes, Fc = calculated from model)
- Contoured at **1.0–1.5 σ** (standard deviations above the mean density)
- The model should sit *inside* this density

**Fo-Fc difference map** (green / red mesh)
- **Green blobs at +3 σ** → electrons in data but not in model → *add atoms here*
- **Red blobs at −3 σ** → atoms in model but no electron support → *remove or move atoms*

```
INTERPRETING DIFFERENCE DENSITY

  GOOD FIT              MISSING ATOMS           WRONG ATOMS
                        (green, +Fo-Fc)          (red, -Fo-Fc)

    ╭──────╮               ╭──────╮                ╭──────╮
   ╱  ○──○  ╲             ╱   ○    ╲              ╱   ○    ╲
  │  ─○──○─  │           │  GREEN   │            │   RED    │
   ╲  ○──○  ╱             ╲  BLOB  ╱              ╲  BLOB  ╱
    ╰──────╯               ╰──────╯                ╰──────╯

 Model inside density    Add atoms at +3σ      Remove/move atoms at -3σ
```

In [ ]:
import urllib.request, os

def download_electron_density(pdb_id, output_dir="."):
    """
    Download 2Fo-Fc and Fo-Fc electron density maps from the Uppsala EDS server.

    Parameters
    ----------
    pdb_id : str
        4-letter PDB accession code (case-insensitive).
    output_dir : str
        Directory to save the .omap files.

    Notes
    -----
    The Uppsala EDS server (eds.bmc.uu.se) provides pre-calculated maps for
    most PDB entries determined by X-ray crystallography.
    """
    pdb_id = pdb_id.lower()
    mid = pdb_id[1:3]
    base = f"https://eds.bmc.uu.se/eds/dfs/{mid}/{pdb_id}"

    targets = {
        f"{pdb_id}.omap":      "2Fo-Fc map",
        f"{pdb_id}_diff.omap": "Fo-Fc difference map",
    }
    os.makedirs(output_dir, exist_ok=True)

    for fname, label in targets.items():
        url  = f"{base}/{fname}"
        dest = os.path.join(output_dir, fname)
        print(f"Downloading {label} → {dest}")
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"  saved ({os.path.getsize(dest):,} bytes)")
        except Exception as exc:
            print(f"  error: {exc}")

# Example — uncomment to run:
# download_electron_density("1crn", output_dir="/tmp/eds_maps")

# ── nglview snippet (requires: pip install nglview) ──────────────────────────
# import nglview as nv
# view = nv.show_pdbid("1crn")
# view.add_component("/tmp/eds_maps/1crn.omap")          # 2Fo-Fc  (blue)
# view.add_component("/tmp/eds_maps/1crn_diff.omap")     # Fo-Fc   (green/red)
# view

### 5.1.2 Resolution and Density Quality

**Resolution** (in Å) describes the finest detail visible in the electron density — set by the highest-angle diffraction data collected. Lower numbers = finer detail.

| Resolution | Features visible | Typical application |
|---|---|---|
| < 1.0 Å | Individual atoms, anisotropic displacement | Ultra-high detail, drug design |
| 1.0–1.5 Å | Nearly all atoms clearly separated | Detailed mechanistic studies |
| 1.5–2.5 Å | Side-chain conformations visible | Standard structural biology |
| 2.5–3.5 Å | Main chain; limited side-chain detail | Large complexes |
| > 3.5 Å | Secondary structure elements only | Virus capsids, very large assemblies |

```
    1.0 Å         2.0 Å         3.0 Å         4.0 Å
   ┌──────┐      ┌──────┐      ┌──────┐      ┌──────┐
   │ ○  ○ │      │  ○○  │      │  ○   │      │      │
   │  ○○  │      │ ○    │      │   ○  │      │  ○   │
   │○    ○│      │    ○ │      │      │      │      │
   └──────┘      └──────┘      └──────┘      └──────┘
  Atomic res.  Atoms merge   Blobs only   Shape only
```

See **Section 5.5** for how R-factor and R-free connect resolution to model quality.

### 5.1.3 Crystal Lattices and Unit Cell

A crystal is a 3-D periodic repetition of a **unit cell** — the smallest parallelepiped that tiles space. Each unit cell is defined by six parameters:

| Parameter | Symbol | Meaning |
|---|---|---|
| Edge lengths | a, b, c | Dimensions in Ångströms |
| Inter-edge angles | α, β, γ | Angles between the edges (degrees) |

The symmetry constraints on these parameters define the **7 crystal systems**:

| System | Constraints | Common space group |
|---|---|---|
| Cubic | a=b=c, α=β=γ=90° | I 4 3 2 |
| Tetragonal | a=b≠c, α=β=γ=90° | P 4 21 2 |
| Orthorhombic | a≠b≠c, α=β=γ=90° | P 21 21 21 |
| Hexagonal | a=b≠c, α=β=90°, γ=120° | P 6 2 2 |
| Trigonal | a=b=c, α=β=γ≠90° | R 3 2 |
| Monoclinic | a≠b≠c, α=γ=90°, β≠90° | P 1 21 1 |
| Triclinic | a≠b≠c, α≠β≠γ≠90° | P 1 |

Unit cell parameters are stored in the **CRYST1** record of every PDB file.

In [ ]:
def parse_cryst1(pdb_file):
    """
    Extract unit cell parameters from the CRYST1 record of a PDB file.

    CRYST1 column layout (fixed-width):
      cols  7-15  a (Å)
      cols 16-24  b (Å)
      cols 25-33  c (Å)
      cols 34-40  alpha (°)
      cols 41-47  beta  (°)
      cols 48-54  gamma (°)
      cols 56-66  space group
      cols 67-70  Z (copies of AU per unit cell)
    """
    with open(pdb_file) as fh:
        for line in fh:
            if line.startswith("CRYST1"):
                return {
                    "a":     float(line[6:15]),
                    "b":     float(line[15:24]),
                    "c":     float(line[24:33]),
                    "alpha": float(line[33:40]),
                    "beta":  float(line[40:47]),
                    "gamma": float(line[47:54]),
                    "spacegroup": line[55:66].strip(),
                    "Z":     int(line[66:70].strip()) if len(line) > 66 else None,
                }
    return None

# ── Demonstrate on a synthetic CRYST1 line ───────────────────────────────────
# Synthetic example of a monoclinic CRYST1 record (P 1 21 1 space group)
import tempfile, os

example_line = "CRYST1   40.960   18.650   22.520  90.00  90.77  90.00 P 1 21 1      2\n"

# Write to a temp file and parse it
with tempfile.NamedTemporaryFile(mode="w", suffix=".pdb", delete=False) as tmp:
    tmp.write(example_line)
    tmp_path = tmp.name

params = parse_cryst1(tmp_path)
os.unlink(tmp_path)

print("CRYST1 record:")
print(example_line.strip())
print()
print(f"Cell dimensions : a={params['a']} Å, b={params['b']} Å, c={params['c']} Å")
print(f"Cell angles     : α={params['alpha']}°, β={params['beta']}°, γ={params['gamma']}°")
print(f"Space group     : {params['spacegroup']}")
print(f"Z               : {params['Z']} asymmetric units per cell")
print()
print(f"Crystal system  : Monoclinic  (a≠b≠c, α=γ=90°, β={params['beta']}°≠90°)")

### 5.1.4 Space Groups and Asymmetric Units

A **space group** encodes all the symmetry operations that map the crystal onto itself (rotations, screw axes, glide reflections, etc.). There are **230 space groups** in total, but proteins are chiral — they cannot have mirror planes or inversion centres. Only **65 chiral (Sohncke) space groups** are therefore accessible to proteins.

**Most common space groups in the PDB:**

| Space group | Crystal system | ~% of PDB |
|---|---|---|
| P 21 21 21 | Orthorhombic | ~24 % |
| P 1 21 1 | Monoclinic | ~15 % |
| C 1 2 1 | Monoclinic | ~8 % |
| P 21 21 2 | Orthorhombic | ~5 % |
| P 43 21 2 | Tetragonal | ~4 % |

The **asymmetric unit (AU)** is the smallest portion of the unit cell from which the entire cell can be regenerated by applying the space-group symmetry operations. Z in the CRYST1 record is the number of AU copies per unit cell.

```
     UNIT CELL  (4-fold rotational symmetry, Z = 4)

   ┌──────────────┬──────────────┐
   │   copy 2     │   copy 3     │
   │   ╔══════╗   │   ╔══════╗   │
   │   ║  P   ║   │   ║  P   ║   │
   │   ╚══════╝   │   ╚══════╝   │
   ├──────────────┼──────────────┤
   │   copy 1     │  ← AU →      │
   │   ╔══════╗   │   ╔══════╗   │
   │   ║  P   ║   │   ║  P   ║   │
   │   ╚══════╝   │   ╚══════╝   │
   └──────────────┴──────────────┘

  Symmetry applies 4× to AU → complete unit cell
```

In [ ]:
import numpy as np

def apply_symmetry_operation(coords, rotation, translation):
    """
    Apply one crystallographic symmetry operation to a set of coordinates.

    Parameters
    ----------
    coords : array-like, shape (3,)
        Fractional or Cartesian coordinates of an atom.
    rotation : array-like, shape (3, 3)
        Rotation matrix from the space-group symmetry operation.
    translation : array-like, shape (3,)
        Translation vector (in the same coordinate system as coords).

    Returns
    -------
    np.ndarray, shape (3,)
        Coordinates of the symmetry-related copy.
    """
    return np.dot(rotation, coords) + translation

# Example: 2-fold screw axis along b  (space group P 1 21 1)
# Symmetry operation: (-x, y + 1/2, -z)
rotation_P21 = np.array([
    [-1,  0,  0],
    [ 0,  1,  0],
    [ 0,  0, -1],
], dtype=float)

translation_P21 = np.array([0.0, 0.5, 0.0])

original = np.array([0.12, 0.34, 0.56])
sym_copy = apply_symmetry_operation(original, rotation_P21, translation_P21)

print("Symmetry operation: (-x, y+½, -z)  [P 1 21 1]")
print(f"Original AU atom   : {original}")
print(f"Symmetry-related   : {sym_copy}")
print()
print("Interpretation:")
print("  x → -x :  reflected across the b–c plane")
print("  y → y+½:  shifted half a unit cell along b (the screw component)")
print("  z → -z :  reflected across the a–b plane")

#### Exercises

**Exercise A — Crystal system identification (**)

The CRYST1 record below is from PDB entry 3J3Q:

```
CRYST1  232.000  232.000  235.000  90.00  90.00 120.00 P 61 2 2     12
```

1. Identify the crystal system (refer to the table in 5.1.3).
2. How many asymmetric units are in the unit cell?
3. Is this space group compatible with a protein? Why or why not?

---

**Exercise B — Difference map interpretation (*)**

During refinement of a 2.0 Å structure you observe:
- A **red blob** at residue Lys 47 NZ (the terminal amine nitrogen).
- A **green blob** 2 Å away from the same position.

What does each blob indicate, and what action should the crystallographer take?

### 5.2 Cryo-Electron Microscopy (cryo-EM)

The "resolution revolution" (~2013–present) transformed cryo-EM from a technique limited to ~6–8 Å into one routinely achieving 2–3 Å, enabled by:
- **Direct electron detectors** (DDD): ~10× more efficient than film or CCD
- **Movie-mode data collection** + beam-induced motion correction
- **Better classification algorithms** (RELION, cryoSPARC)

**Single-particle analysis (SPA) workflow:**
```
Vitrified sample (flash-frozen in thin amorphous ice)
    ↓  TEM imaging (80–300 kV)
Movie frames (electron micrographs)
    ↓  Motion correction, CTF estimation
Corrected micrographs
    ↓  Particle picking (automated)
Particle stack (10⁴–10⁶ particles)
    ↓  2D classification (remove junk)
    ↓  3D classification & refinement
3D electron density map
    ↓  Atomic model building & real-space refinement
Final structure
```

| Feature | Cryo-EM | X-ray crystallography |
|---------|---------|----------------------|
| Sample state | Near-native, no crystal required | Crystal required |
| Molecular weight | Optimal > 100 kDa; difficult < 50 kDa | No lower limit |
| Conformational flexibility | Multiple conformers from one dataset | Crystal packing constrains motion |
| Resolution | Typically 2–4 Å; routinely 1.5–2 Å for best cases | Can reach 0.8 Å (sub-atomic) |
| Throughput | Automated data collection (Talos Arctica, Titan Krios) | Synchrotron beamlines |
| Phase problem | No phase problem (direct inversion) | Requires phase determination |

Cryo-EM is now the **method of choice** for large complexes (ribosomes, proteasomes, membrane proteins, filaments) and molecules that resist crystallization.

### 5.3 NMR Spectroscopy

Solution NMR determines protein structure in solution and gives access to **dynamics** that crystal structures miss.

**Principle:** Nuclei with non-zero spin (¹H, ¹³C, ¹⁵N) have quantized energy levels in a magnetic field. RF pulses excite nuclei; the free induction decay (FID) is Fourier-transformed to give the NMR spectrum.

**Key observables:**
| Observable | Information |
|-----------|-------------|
| **Chemical shift** (δ, ppm) | Local electronic environment — secondary structure, solvent exposure, ligand binding |
| **NOE** (Nuclear Overhauser Effect) | Distance between protons < 5 Å — primary source of distance restraints for structure calculation |
| **J-coupling** | Torsion angles (φ, ψ via ³J_HNHα Karplus relationship) |
| **Relaxation** (T₁, T₂, NOE) | Protein dynamics: ns–ps motions (T₁/T₂ ratio ∝ τ_c ∝ MW) |
| **RDC** (Residual dipolar coupling) | Long-range structural restraints from partial alignment |

**Structure calculation pipeline:**
1. Assign backbone resonances (HNCA, HNCaCb, HNCO — triple resonance experiments)
2. Assign side chain resonances (HCCH-TOCSY, HCCH-COSY)
3. Assign NOE cross-peaks → distance restraints
4. Calculate structure ensemble (CYANA, CNS, XPLOR-NIH): typically 20 lowest-energy conformers deposited

**Size limitation:** Line broadening from slow tumbling. Practical upper limit ~50 kDa with TROSY-type experiments and ²H/¹³C/¹⁵N labeling; TROSY can push to ~100 kDa for some systems.

### 5.4 Method Comparison and Choosing the Right Approach

| Criterion | X-ray | Cryo-EM | NMR |
|-----------|-------|---------|-----|
| Resolution (routinely achievable) | 1.5–3 Å | 2–4 Å | ~2 Å (ensemble) |
| Sample size optimal | Any (crystals needed) | > 100 kDa preferred | < 50 kDa preferred |
| Crystal required | Yes | No | No |
| Dynamics information | Limited (B-factor only) | Limited | Rich (ps–μs timescales) |
| Solution conditions | Crystal packing effect | Near-native | Near-native |
| Sample amount | ~1–10 mg (for crystal) | ~0.1–0.5 mg | ~0.5–2 mM in ~0.5 mL |
| Ligand binding study | Yes (co-crystallization or soaking) | Yes (particle classification) | Excellent (CSP, KD from titration) |
| Membrane proteins | Difficult (detergent) | Excellent (nanodiscs) | Possible (bicelles) |

**Decision rules:**
- Small stable protein, need atomic detail → **X-ray**
- Large complex, membrane protein, difficult to crystallize → **cryo-EM**
- Dynamics, ligand screening, solution behavior → **NMR**
- No experimental structure available → **AlphaFold2** prediction (but validate before trusting drug design)

In [ ]:
# Parse and interpret PDB header fields relevant to structure quality
# (Works on any PDB file downloaded from https://www.rcsb.org/)

EXAMPLE_HEADER = """
HEADER    HYDROLASE                               23-FEB-98   1AKE
EXPDTA    X-RAY DIFFRACTION
REMARK   2 RESOLUTION.    2.00 ANGSTROMS.
REMARK   3   R VALUE            (WORKING SET) : 0.196
REMARK   3   FREE R VALUE                     : 0.229
REMARK   3   FREE R VALUE TEST SET SIZE (%)   : 5.100
CRYST1   69.850   69.850  103.180  90.00  90.00  90.00 P 41 21 2    8
"""


def parse_pdb_header(pdb_text: str) -> dict:
    """Extract key quality metrics from PDB header text."""
    info = {}
    for line in pdb_text.splitlines():
        if line.startswith('EXPDTA'):
            info['experiment'] = line[10:].strip()
        elif 'RESOLUTION.' in line:
            parts = line.split()
            for i, p in enumerate(parts):
                if p == 'ANGSTROMS.' and i > 0:
                    try:
                        info['resolution_A'] = float(parts[i - 1])
                    except ValueError:
                        pass
        elif 'R VALUE' in line and 'WORKING' in line:
            parts = line.split()
            try:
                info['R_work'] = float(parts[-1])
            except ValueError:
                pass
        elif 'FREE R VALUE' in line and 'SIZE' not in line and 'SET' not in line:
            parts = line.split()
            try:
                info['R_free'] = float(parts[-1])
            except ValueError:
                pass
        elif line.startswith('CRYST1'):
            parts = line.split()
            if len(parts) >= 7:
                info['unit_cell'] = {
                    'a': float(parts[1]), 'b': float(parts[2]), 'c': float(parts[3]),
                    'alpha': float(parts[4]), 'beta': float(parts[5]), 'gamma': float(parts[6]),
                }
            if len(parts) >= 8:
                info['space_group'] = ' '.join(parts[7:-1])
    return info


def evaluate_structure_quality(info: dict) -> None:
    """Print quality assessment for a parsed PDB header."""
    print("=== Structure Quality Assessment ===")
    print(f"Experiment type: {info.get('experiment', 'Unknown')}")
    res = info.get('resolution_A')
    if res:
        if res < 1.5:
            qual = 'Excellent (atomic resolution)'
        elif res < 2.5:
            qual = 'Good'
        elif res < 3.5:
            qual = 'Acceptable'
        else:
            qual = 'Low — interpret side chains with caution'
        print(f"Resolution: {res} Å — {qual}")
    r_work = info.get('R_work')
    r_free = info.get('R_free')
    if r_work and r_free:
        print(f"R-work: {r_work:.3f} ({'OK' if r_work < 0.25 else 'HIGH'})")
        print(f"R-free: {r_free:.3f} ({'OK' if r_free < 0.30 else 'HIGH'})")
        diff = r_free - r_work
        overfitting = 'WARNING: possible overfitting' if diff > 0.10 else 'OK'
        print(f"R_free - R_work = {diff:.3f} ({overfitting})")
    if 'space_group' in info:
        print(f"Space group: {info['space_group']}")
    if 'unit_cell' in info:
        uc = info['unit_cell']
        print(f"Unit cell: {uc['a']:.2f} × {uc['b']:.2f} × {uc['c']:.2f} Å, "
              f"{uc['alpha']:.1f}° {uc['beta']:.1f}° {uc['gamma']:.1f}°")


info = parse_pdb_header(EXAMPLE_HEADER)
evaluate_structure_quality(info)

### 5.5 Quality Indicators in PDB Structures

From the ФББ Bioinformatics 3D course — quality indicators for X-ray models:

| Indicator | What it measures | Good value |
|-----------|-----------------|------------|
| **Resolution** | Experimental data quality | < 2.5 Å for most work |
| **R-factor** | Model vs. experimental |Fobs| | < 25% |
| **R-free** | Same on held-out reflections (5%) | < 30%; R_free − R < 10% |
| **Ramachandran plot** | Backbone geometry | > 90% in favored regions |
| **Rotamer outliers** | Side chain geometry | < 1% (good resolution); < 7% (low resolution) |
| **B-factor (isotropic)** | Atomic displacement reliability | 10–20 Å² good; > 60 Å² — don't interpret that atom |
| **Occupancy** | Fraction of unit cells with atom at that position | = 1.0 (normal); < 1.0 for disordered residues |
| **RSR (Real-Space R)** | Local model vs. experimental electron density | < 10% good; > 20% suspicious |
| **Clash score** | Steric clashes between non-bonded atoms | < 10 (Molprobity) |

---
## Exercises

### Exercise 1: Ion Mass Calculation
A peptide has sequence `PEPTIDER`. Calculate:
- (a) Its neutral monoisotopic mass
- (b) The m/z of the [M+2H]²⁺ ion (ESI)
- (c) The b3 and y5 ion masses
- (d) What mass difference distinguishes a leucine from an isoleucine substitution in a b/y ion? What does this imply for de novo sequencing?

In [ ]:
# Exercise 1 workspace
seq_ex1 = "PEPTIDER"

# (a) Neutral mass
mass_neutral = peptide_mass(seq_ex1)
print(f"(a) Neutral mass of {seq_ex1}: {mass_neutral:.4f} Da")

# (b) [M+2H]2+
mz_2plus = (mass_neutral + 2 * H) / 2
print(f"(b) [M+2H]²⁺: {mz_2plus:.4f} Da")

# (c) b3 and y5
b_ions_ex1, y_ions_ex1 = by_ions(seq_ex1)
print(f"(c) b3 ion mass: {b_ions_ex1[2]:.4f} Da")
print(f"    y5 ion mass: {y_ions_ex1[2]:.4f} Da")

# (d) L vs I
print(f"\n(d) Mass of L = {AA_MONO_MASS['L']:.5f} Da")
print(f"    Mass of I = {AA_MONO_MASS['I']:.5f} Da")
print(f"    Difference = {abs(AA_MONO_MASS['L'] - AA_MONO_MASS['I']):.5f} Da")
print("    L and I are isobaric — indistinguishable by MS/MS alone.")
print("    Requires ETD/ECD or chemical derivatization to distinguish.")

### Exercise 2: Trypsin Digestion Analysis
Use the provided `trypsin_digest` function on the following protein sequence. Then:
- (a) How many peptides are generated with 0 missed cleavages? With 1?
- (b) What fraction of the protein sequence is covered by peptides with mass 800–3000 Da (the optimal LC-MS/MS range)?
- (c) Which peptide has the highest mass? Would it be detectable by a standard Orbitrap method (m/z range 200–2000)?

In [ ]:
# Exercise 2 workspace
ex2_protein = (
    "MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSY"
    "RKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHQRQEI"
    "KRVKDSEDVPMVLVGNKCDLAARTVESRQAQDLARSYGIPYIETSAKTRQHVREVDRE"
)  # Human KRAS (truncated)

pep_0mc = trypsin_digest(ex2_protein, missed_cleavages=0)
pep_1mc = trypsin_digest(ex2_protein, missed_cleavages=1)
print(f"(a) Peptides (0 MC): {len(pep_0mc)},  Peptides (1 MC): {len(pep_1mc)}")

# (b) Coverage by detectable peptides
detected_seqs = [p for p in pep_0mc if 800 <= peptide_mass(p) <= 3000
                 and all(aa in AA_MONO_MASS for aa in p)]
covered_aa = set()
for pep in detected_seqs:
    idx = ex2_protein.find(pep)
    if idx >= 0:
        covered_aa.update(range(idx, idx + len(pep)))
coverage = len(covered_aa) / len(ex2_protein)
print(f"(b) Peptides in 800–3000 Da range: {len(detected_seqs)}")
print(f"    Sequence coverage: {coverage*100:.1f}%")

# (c) Highest mass peptide
valid_peps = [(p, peptide_mass(p)) for p in pep_0mc
              if all(aa in AA_MONO_MASS for aa in p)]
heaviest_pep, heaviest_mass = max(valid_peps, key=lambda x: x[1])
# Minimum charge state needed to fall within m/z 200-2000
min_z = int(np.ceil((heaviest_mass + H) / (2000 - H)))
print(f"\n(c) Heaviest peptide: {heaviest_pep}")
print(f"    Mass: {heaviest_mass:.2f} Da")
print(f"    Minimum charge z={min_z} to appear at m/z < 2000 → "
      f"detectable={'Yes' if min_z <= 5 else 'Borderline'}")

### Exercise 3: FDR Estimation

A database search returns 500 PSMs above score threshold T from a target database and 8 PSMs from a decoy database at the same threshold.

(a) Estimate the FDR at threshold T.  
(b) If we raise the threshold so only 200 target PSMs remain and 1 decoy PSM, what is the new FDR?  
(c) Why does the decoy FDR formula overestimate the true FDR? (Hint: think about how decoy hits behave compared to target false positives.)

In [ ]:
# Exercise 3
def estimate_fdr(target_hits: int, decoy_hits: int) -> float:
    return decoy_hits / target_hits if target_hits > 0 else 1.0

fdr_a = estimate_fdr(500, 8)
fdr_b = estimate_fdr(200, 1)

print(f"(a) FDR at loose threshold: {fdr_a*100:.2f}% ({500} target, {8} decoy)")
print(f"(b) FDR at strict threshold: {fdr_b*100:.2f}% ({200} target, {1} decoy)")
print()
print("(c) The decoy FDR formula can OVERESTIMATE because:")
print("    - Reversed sequences sometimes match spectral noise as efficiently as true hits")
print("      (especially for short peptides), inflating the decoy count.")
print("    - It can also UNDERESTIMATE if decoy peptides share mass with target peptides")
print("      (especially after modifications), causing decoy hits to 'compete away'")
print("      from false target hits.")
print("    The formula assumes decoy hits ≈ target false positives — a good approximation")
print("    but not exact. Peptide-level FDR is more conservative than PSM-level FDR.")

### Exercise 4: Conservation Analysis

You are engineering a lipase for improved activity in organic solvents. From a published MSA of 50 homologous lipases, you compute conservation scores. Positions with mutability > 0.6 are on the protein surface. Positions 45 and 67 have mutability 0.08 (part of the catalytic triad Ser-His-Asp). Position 112 has mutability 0.75 and is surface-exposed.

(a) Which positions would you **not** mutate and why?  
(b) For position 112, what experimental approach would you use to find the optimal amino acid?  
(c) You want to improve thermostability without losing activity. What general strategy would you apply, and which positions would you prioritize?

In [ ]:
# Exercise 4 — conceptual answer
print("(a) Positions 45 and 67 (catalytic triad, mutability=0.08) should NOT be mutated.")
print("    These are Ser-His-Asp — fully conserved across the family because any substitution")
print("    destroys catalytic activity. Shannon entropy ≈ 0 for these positions.")
print()
print("(b) For position 112 (mutability=0.75, surface-exposed):")
print("    → Saturation mutagenesis: synthesize all 20 single amino acid variants at pos 112")
print("    → Express and screen for activity in organic solvent (e.g., 20–50% acetonitrile)")
print("    → This gives a complete picture with only 20 experiments")
print("    → Alternatively: CAST (combinatorial active-site testing) if combining with nearby positions")
print()
print("(c) Thermostability engineering strategy:")
print("    1. Identify flexible loops from B-factors in crystal structure (high B = flexible)")
print("    2. These loops are good targets: mutations that rigidify them (e.g., Pro at i+1")
print("       position, Gly→Ala, disulfide bonds between loops) typically increase Tm")
print("    3. Use FoldX/Rosetta to predict ΔΔG for Ala scanning of loop residues:")
print("       negative ΔΔG (stabilizing) candidates are prioritized")
print("    4. Avoid positions in or near the catalytic triad")
print("    5. Validate top 5–10 predicted stabilizing variants experimentally (DSF, Tm shift)")

### Exercise 5: Choosing a Structural Method

For each of the following scenarios, select the most appropriate experimental method (X-ray, cryo-EM, NMR) and justify your choice:

(a) A 15 kDa single-domain antibody fragment (nanobody) — you need atomic resolution to guide rational engineering.  
(b) A 450 kDa AAA+ ATPase that cycles between open and closed conformations during ATP hydrolysis — you want to capture both states.  
(c) A membrane protein embedded in lipid nanodisc — the nanodisc assembly is ~120 kDa total.  
(d) A 25 kDa enzyme — you want to measure its binding affinity ($K_d$) for a small-molecule inhibitor and map which residues are affected.  
(e) Any protein — you have no budget for crystallization or cryo-EM, but you have a homologous structure at 40% sequence identity in PDB.

In [ ]:
# Exercise 5 — answer key
answers = [
    ("(a) 15 kDa nanobody, atomic resolution needed",
     "X-ray crystallography",
     "Small and stable → crystallizes well. Routinely achieves <2 Å for nanobodies. "
     "NMR is feasible but de novo assignment at 15 kDa is labor-intensive."),

    ("(b) 450 kDa AAA+ ATPase, multiple conformations",
     "Cryo-EM",
     "Large complex ideal for cryo-EM. 3D classification can separate open/closed states "
     "from a single dataset without needing separate crystal forms."),

    ("(c) Membrane protein in nanodisc, ~120 kDa",
     "Cryo-EM",
     "Nanodiscs resist crystallization and are too large/dynamic for NMR. "
     "Cryo-EM of membrane proteins in nanodiscs is now routine (TRPV1, GluCl, etc.)."),

    ("(d) 25 kDa enzyme, ligand binding mapping",
     "NMR",
     "15N-HSQC chemical shift perturbation (CSP) experiment: assign backbone, titrate inhibitor, "
     "map perturbed residues onto structure. Gives Kd from intensity or shift vs. [ligand]. "
     "X-ray is also possible but requires co-crystallization and cannot give Kd directly."),

    ("(e) No experimental resources, 40% identity homolog in PDB",
     "Homology modeling (or AlphaFold2 prediction)",
     "At 40% identity, homology modeling is reliable (RMSD typically 1–2 Å for core). "
     "Use SWISS-MODEL, Modeller, or AlphaFold2. Not an experimental structure — "
     "validate with Ramachandran, Molprobity, and biologically test predictions."),
]

for scenario, method, rationale in answers:
    print(f"{scenario}")
    print(f"  Best method: {method}")
    print(f"  Rationale:   {rationale}")
    print()

---
## Summary

| Topic | Key concepts | Python tools covered |
|-------|-------------|---------------------|
| **MS ionization** | MALDI vs ESI, m/z = (M + zH)/z | `peptide_mass`, charge state formula |
| **MS analyzers** | TOF, Orbitrap (high res), ion trap (fast) | — |
| **MS/MS** | b/y ion series, CID/HCD fragmentation | `by_ions` |
| **Bottom-up workflow** | Digestion → LC → DDA → database search | `trypsin_digest` |
| **PMF** | Mass fingerprinting, mass tolerance matching | `simple_pmf_search` |
| **FDR estimation** | Target-decoy, PSM/peptide/protein levels | `estimate_fdr` |
| **Quantification** | LFQ, iBAQ, SILAC, TMT | Median normalization, volcano/MA plots |
| **DDA vs DIA** | Stochastic vs. reproducible acquisition | — |
| **Directed evolution** | epPCR, DNA shuffling, saturation mutagenesis | — |
| **Conservation scoring** | Shannon entropy, mutability score | `shannon_entropy`, `msa_conservation` |
| **ΔΔG prediction** | FoldX/Rosetta, stabilizing vs. destabilizing | Simulated scanning mutagenesis |
| **X-ray crystallography** | Bragg's law, R-factor, phase problem, MR | `parse_pdb_header`, `evaluate_structure_quality` |
| **Electron density maps** | 2Fo-Fc / Fo-Fc coefficients, σ contouring, positive/negative blobs | `download_electron_density` |
| **Resolution & quality** | Å scale, Rfree/Rwork, real-space CC, B-factors | — |
| **Crystal lattices** | Unit cell (a,b,c,α,β,γ), 7 crystal systems, CRYST1 record | `parse_cryst1` |
| **Space groups** | 230 total, 65 Sohncke (chiral), asymmetric unit, Z value | `apply_symmetry_operation` |
| **Cryo-EM** | SPA workflow, resolution revolution, vs. X-ray | — |
| **NMR** | Chemical shifts, NOEs, dynamics, size limit | — |

**Next steps:**
- Apply MaxQuant/Perseus to a real proteomics dataset (PRIDE repository, `PXD` accessions)
- Use FoldX `BuildModel` to actually compute ΔΔG on a PDB file
- Run a ConSurf job on your protein family MSA to get residue-level conservation scores
- Explore AlphaFold2 predictions via the EBI server or run locally with ColabFold